# 🌍 Spatial-LLM — Per-module fusion gating A/B (Kaggle)

Tests the **synchronization** idea: does giving each spatial module (coord/elevation,
grid cells, place-memory, tile) its **own** fusion gate beat a single **shared** gate
on the elevation task — and do the learned gates show the model leaning on the
elevation module?

**Arms (same data, same 3D coords — only the gating differs):**
- `shared`  → `configs/coord_3d.yaml`        (one gate for all modules — baseline)
- `permod`  → `configs/coord_3d_permod.yaml` (one gate per module — treatment)

### ⚠️ Read before running
- **GPU:** Settings → Accelerator → **GPU T4 x2** (or P100), **Internet ON**.
- Training **pins to a single GPU** (trainer.py sets `CUDA_VISIBLE_DEVICES=0`). The
  model isn't DataParallel-safe, so on T4 x2 it would otherwise hang — this is the fix
  for the v1 timeout.
- **Start with `SEEDS = [42]`** (cell 4) to validate end-to-end in ~10–20 min before
  committing to the full 3-seed sweep.
- For the long sweep, use **Save Version → Save & Run All (Commit)** so it runs in the
  background and survives closing the tab. Each run writes its own results, so a later
  failure never loses earlier runs.

## 1. Setup — clone/pull latest `main` + install deps + pre-download model

In [ ]:
%cd /kaggle/working
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'        # Xet streaming backend stalls on Kaggle
os.environ['CUDA_VISIBLE_DEVICES'] = '0'      # single GPU (model isn't DataParallel-safe)
if not os.path.isdir('Spatial-LLM'):
    !git clone https://github.com/Mohammadzamanid/Spatial-LLM.git
%cd /kaggle/working/Spatial-LLM
!git pull origin main          # pull the per-module gating + single-GPU fix
!pip install -q transformers peft accelerate datasets bitsandbytes
!pip install -q geopandas shapely mercantile pyyaml pillow
import torch
print('commit below (want f5f519e or later):')
!git log --oneline -1
print('GPUs visible to notebook:', torch.cuda.device_count())
# Pre-download the base model ONCE so training loads from local cache instead of a
# live stream that stalls mid-materialization. ~3 GB; shows a real download bar.
from huggingface_hub import snapshot_download
print('model cached at:', snapshot_download('Qwen/Qwen2.5-1.5B'))

## 2. Data — elevation task
Both arms train on the *same* file; `coord_2d` vs `coord_3d` differ only in whether the
model is *given* elevation. Split is fixed (data seed=42), so every run shares one
held-out val set — clean for an A/B. (~4.5k cities have elevation, so expect ~3.8k
train / ~700 val.)

In [ ]:
!python -m src.data.real_datasets --dataset cities15000 --task elevation --n_train 8000 --n_val 1000
import json
with open('data/processed/val.jsonl') as f:
    r = json.loads(f.readline())
print('Q:', r['question']); print('A:', r['answer'], '| elevation:', r.get('elevation'))

## 3. Config — arms & seeds (shared by the train/eval cells)

In [ ]:
ARMS = {'shared': 'configs/coord_3d.yaml',          # baseline: one shared gate
        'permod': 'configs/coord_3d_permod.yaml'}    # treatment: per-module gates
SEEDS = [42]          # validate first; then rerun cells 4-5 with [42, 43, 44]
print('arms:', list(ARMS), '| seeds:', SEEDS)

## 4. Train
~10–20 min per run on a T4. With `SEEDS=[42]` that's 2 runs; the full 3-seed sweep is 6.

In [ ]:
import subprocess, os
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0',   # single GPU (belt + suspenders)
       'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
       'TOKENIZERS_PARALLELISM': 'false'}
for arm, cfg in ARMS.items():
    for s in SEEDS:
        out = f'outputs/{arm}_seed{s}'
        print(f'\n===== TRAIN {arm} seed={s} -> {out} =====', flush=True)
        subprocess.run(['python','-m','src.training.trainer',
                        '--config',cfg,'--seed',str(s),'--output_dir',out],
                       check=True, env=env)

## 5. Evaluate — balanced accuracy + the gate read-out
`--dump-gates` prints each layer's per-module gate (tanh). Hypothesis: on the elevation
task the **coord/elev** gate opens wide while **grid** stays near zero.

In [ ]:
import subprocess, os, json, re, statistics as st
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0', 'TOKENIZERS_PARALLELISM': 'false'}
results = []
for arm, cfg in ARMS.items():
    for s in SEEDS:
        out, label = f'outputs/{arm}_seed{s}', f'{arm}_seed{s}'
        r = subprocess.run(['python','-m','src.eval.accuracy','--config',cfg,
                            '--checkpoint',out,'--val','data/processed/val.jsonl',
                            '--dump-gates','--seed',str(s),'--label',label],
                           capture_output=True, text=True, env=env)
        print(r.stdout[-1600:])
        m = re.search(r'===RESULT-JSON-START===\n(.*)\n===RESULT-JSON-END===', r.stdout, re.S)
        if m: results.append(json.loads(m.group(1)))
        else: print('⚠️ no result block for', label, '\n', r.stderr[-800:])

print('\n================ SUMMARY (balanced accuracy) ================')
by_arm = {}
for x in results:
    by_arm.setdefault(x['label'].split('_seed')[0], []).append(x['balanced_accuracy'])
for arm, accs in by_arm.items():
    accs = [a for a in accs if a is not None]
    mean = st.mean(accs); sd = st.pstdev(accs) if len(accs) > 1 else 0.0
    print(f'  {arm:8s} {mean:.3f} ± {sd:.3f}  (n={len(accs)}: {[round(a,3) for a in accs]})')

## 6. Copy this block back into the chat
Paste the `===PASTE-THIS-BACK===` block; it gets committed to
`results/per_module_gating.json` on `main` (a copy is also saved to Kaggle output).

In [ ]:
import json, os
payload = {'experiment':'per_module_gating','task':'elevation',
           'dataset':'cities15000','metric':'balanced_accuracy',
           'source':'kaggle','reproduced_in_repo':True,'seeds':SEEDS,'runs':results}
os.makedirs('/kaggle/working/results', exist_ok=True)
with open('/kaggle/working/results/per_module_gating.json','w') as f:
    json.dump(payload, f, indent=2)
print('saved -> /kaggle/working/results/per_module_gating.json')
print('\n===PASTE-THIS-BACK===')
print(json.dumps(payload, indent=2))
print('===END===')